# UtilityRAG: Retrieval-Augmented Generation for GIS Conversion Docs

**GeoAI 582 Final Project · Maxx Millstein**

A small LangChain RAG system that answers GIS conversion questions from internal documentation, cites the source file, and says "not in the docs" when it can't find an answer. A 10-question test compares it against the same LLM without retrieval.

The pipeline follows LangChain's [rag-from-scratch](https://github.com/langchain-ai/rag-from-scratch) notebooks (Parts 1–4, plus Multi-Query from Part 5), adapted to Claude + Voyage AI:

| This notebook | rag-from-scratch |
|---|---|
| Part 1: Overview (whole pipeline in one cell) | Part 1 |
| Part 2: Indexing (load, split by heading, embed, store) | Part 2 |
| Part 3: Retrieval | Part 3 |
| Part 4: Generation (cited answers + abstention) | Part 4 |
| Part 5: Baseline (LLM alone) | — |
| Part 6: Evaluation | — |
| Part 7 (optional): Multi-Query retrieval | Part 5 |

**Corpus:** 12 synthetic Markdown docs in `docs/` for a fictional conversion team (GridLine Data Services) and three fictional cooperatives. Key values (field names, codes, tolerances) are deliberately nonstandard so the LLM can't guess them, and the test measures retrieval, not prior knowledge.

The same pipeline is packaged as scripts for the repo: `ingest.py`, `ask.py`, `evaluate.py`, all built on `rag_core.py`.

## Environment
### (1) Packages

In [1]:
# Run once in the RAG_Project conda env
! pip install langchain langchain-community langchain-anthropic langchain-voyageai langchain-chroma langchain-text-splitters langsmith tiktoken numpy pandas python-dotenv

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain-1.4.7-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_anthropic-1.7.6-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_voyageai-0.4.1-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_text_splitters-1.1.3-py3-none-any.whl.metadata (3.6 kB)
  Using cached langsmith-0.14.12-py3-none-any.whl.metadata (38 kB)
  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
  Using cached langchain_core-1.7.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

### (2) API keys

Put these in a `.env` file next to the notebook (it's in `.gitignore`):

```
ANTHROPIC_API_KEY=sk-ant-...
VOYAGE_API_KEY=pa-...
# Optional: LangSmith tracing
LANGCHAIN_API_KEY=lsv2_pt_...
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=utilityrag
```

`load_keys()` also falls back to the `*_api_key.txt` files in this folder.

In [2]:
import rag_core as rc

rc.load_keys()

{'ANTHROPIC_API_KEY': True, 'VOYAGE_API_KEY': True, 'LANGCHAIN_API_KEY': True}

### (3) Models

- **LLM:** Claude Haiku 5.5. No `temperature` argument: the Claude API stopped accepting sampling parameters on 2026-10-21.
- **Embeddings:** Voyage `voyage-3.5-lite` (the proposal planned a local Hugging Face model; Voyage is the embedding provider recommended for Claude and needs no GPU).

In [3]:
llm = rc.get_llm()
embd = rc.get_embeddings()
print(rc.LLM_MODEL, "|", rc.EMBED_MODEL)

claude-haiku-5-5 | voyage-3.5-lite


C:\Users\mills\miniconda3\envs\RAG_Project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1: Overview

The whole pipeline in one cell, like the rag-from-scratch quickstart. The loader reads the local `docs/` folder instead of a web page. At work this would be a `ConfluenceLoader`; nothing after it changes.

In [4]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

#### INDEXING ####

# Load
docs = [Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": p.name})
        for p in sorted(Path("docs").glob("*.md"))]

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed (in-memory collection, separate from the main index)
quickstart_store = Chroma.from_documents(documents=splits, embedding=embd,
                                         collection_name="quickstart")
retriever = quickstart_store.as_retriever()

#### RETRIEVAL and GENERATION ####

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
{context}

Question: {question}""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What snapping tolerance should I use for Prairie Wind Rural Electric?")

'Use a snapping tolerance of **0.25 ft** for Prairie Wind Rural Electric. The PWRE conversion notes say the QA lead approved this exception because the network was digitized from paper, and it applies to all PWRE feature classes. It overrides the Revision 7 default of 0.04 ft.'

## Part 2: Indexing

### Load the corpus

In [5]:
docs = rc.load_docs()
for d in docs:
    print(f"{d.metadata['source']:<36} {len(d.page_content):>5} chars")

00_conversion_overview.md             1867 chars
01_target_schema.md                   1888 chars
02_domain_codes.md                    1031 chars
03_coordinate_systems.md              1112 chars
04_qa_thresholds.md                   1353 chars
05_attribute_mapping_general.md       1296 chars
06_coop_bluestem_valley.md            1063 chars
07_coop_ozark_ridge.md                1114 chars
08_coop_prairie_wind.md               1013 chars
09_conversion_tools.md                1332 chars
10_delivery_and_signoff.md             752 chars
11_troubleshooting.md                 1367 chars


### Count tokens

`cl100k_base` is OpenAI's tokenizer, so this is an approximation for Claude/Voyage. It's fine for sizing chunks.

The whole corpus is small enough to fit in one prompt. RAG matters at Confluence scale (hundreds of pages), which the report discusses.

In [6]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str = "cl100k_base") -> int:
    return len(tiktoken.get_encoding(encoding_name).encode(string))

total = sum(num_tokens_from_string(d.page_content) for d in docs)
print(f"Corpus: ~{total:,} tokens")

Corpus: ~3,876 tokens


### Split by heading

The quickstart splits every 1,000 characters, which can cut a table in half or separate a value from the heading that says which cooperative it applies to. Conversion docs are organized by heading, so we split on Markdown headings first (`MarkdownHeaderTextSplitter`), then by size only if a section is still too long. Each chunk keeps its file name and heading path for citations.

In [7]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [("#", "h1"), ("##", "h2"), ("###", "h3")]
header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)

# Example: one doc
for s in header_splitter.split_text(docs[8].page_content):
    print(s.metadata, "|", len(s.page_content), "chars")

{'h1': 'Prairie Wind Rural Electric (PWRE) - Conversion Notes'} | 125 chars
{'h1': 'Prairie Wind Rural Electric (PWRE) - Conversion Notes', 'h2': 'Source System'} | 191 chars
{'h1': 'Prairie Wind Rural Electric (PWRE) - Conversion Notes', 'h2': 'Dual Voltage System'} | 274 chars
{'h1': 'Prairie Wind Rural Electric (PWRE) - Conversion Notes', 'h2': 'QA Exceptions', 'h3': 'Snapping Tolerance'} | 292 chars
{'h1': 'Prairie Wind Rural Electric (PWRE) - Conversion Notes', 'h2': 'QA Exceptions', 'h3': 'Orphan Service Points'} | 128 chars


In [8]:
# Full corpus (rc.split_docs = header split + size split + source/section metadata)
chunks = rc.split_docs(docs)
print(f"{len(docs)} docs -> {len(chunks)} chunks")
print(chunks[0].metadata)
print(chunks[0].page_content[:300])

12 docs -> 55 chunks
{'h1': 'GridLine Conversion Program Overview', 'source': '00_conversion_overview.md', 'section': 'GridLine Conversion Program Overview'}
# GridLine Conversion Program Overview  
GridLine Data Services converts legacy electric distribution GIS data from member cooperatives into the GridLine Electric Target Model (GETM). This page describes the conversion phases and where to find the detailed standards. All cooperatives, codes, and val


### Embeddings

As in rag-from-scratch Part 2: embed a question and a passage, and compare with cosine similarity.

In [9]:
import numpy as np

question = "What snapping tolerance does Prairie Wind use?"
passage = [c for c in chunks if c.metadata["source"] == "08_coop_prairie_wind.md"
           and "Snapping" in c.metadata["section"]][0].page_content
unrelated = [c for c in chunks if c.metadata["source"] == "10_delivery_and_signoff.md"][0].page_content

q_vec = embd.embed_query(question)
p_vec, u_vec = embd.embed_documents([passage, unrelated])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("dimensions:", len(q_vec))
print(f"question vs Prairie Wind snapping section: {cosine_similarity(q_vec, p_vec):.3f}")
print(f"question vs delivery doc:                  {cosine_similarity(q_vec, u_vec):.3f}")

dimensions: 1024
question vs Prairie Wind snapping section: 0.568
question vs delivery doc:                  0.233


### Store in Chroma

Persisted to `chroma_db/` so `ask.py` and `evaluate.py` can reuse it. Re-running this cell replaces the collection instead of adding duplicate chunks.

In [10]:
vectorstore = rc.build_vectorstore(chunks, embd)
print(vectorstore._collection.count(), "chunks stored")

55 chunks stored


## Part 3: Retrieval

Retrieve the top `k = 4` chunks. The scores show whether the cooperative exception outranks the general default, which is the main retrieval challenge in this corpus.

In [11]:
question = "What snapping tolerance should I use for Prairie Wind Rural Electric?"

for doc, dist in vectorstore.similarity_search_with_score(question, k=rc.TOP_K):
    print(f"{dist:.3f}  {doc.metadata['source']:<32} {doc.metadata['section']}")

0.860  08_coop_prairie_wind.md          Prairie Wind Rural Electric (PWRE) - Conversion Notes > QA Exceptions > Snapping Tolerance
0.900  04_qa_thresholds.md              QA Suite Thresholds (Revision 7) > Connectivity Checks
1.115  04_qa_thresholds.md              QA Suite Thresholds (Revision 7) > Attribute Checks
1.116  08_coop_prairie_wind.md          Prairie Wind Rural Electric (PWRE) - Conversion Notes


In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k": rc.TOP_K})
docs_retrieved = retriever.invoke(question)
len(docs_retrieved)

4

## Part 4: Generation

The prompt adds three rules to the rag-from-scratch prompt:

1. Cite the source file after each fact.
2. Cooperative pages override general standards.
3. If the context doesn't contain the answer, say exactly **"not in the docs"**.

Each chunk is labeled with its file and section so the model has something to cite.

In [13]:
print(rc.RAG_TEMPLATE)

You are UtilityRAG, an assistant for GIS conversion analysts at GridLine Data Services.

Answer the question using ONLY the context below. Rules:
- Cite the source file in square brackets after each fact, e.g. [04_qa_thresholds.md].
- Cooperative-specific pages override the general standards when they conflict.
- If the context does not contain the answer, reply with exactly: {abstain}
- Do not use outside knowledge to fill gaps. Keep the answer short.

Context:
{context}

Question: {question}



In [14]:
print(rc.format_docs(docs_retrieved[:2]))

[source: 08_coop_prairie_wind.md | section: Prairie Wind Rural Electric (PWRE) - Conversion Notes > QA Exceptions > Snapping Tolerance]
## QA Exceptions  
### Snapping Tolerance  
Because the PWRE network was digitized from paper, endpoints are rarely coincident. PWRE uses a **snapping tolerance of 0.25 ft** instead of the Revision 7 default. This exception was approved by the QA lead and applies to all PWRE feature classes.

---

[source: 04_qa_thresholds.md | section: QA Suite Thresholds (Revision 7) > Connectivity Checks]
## Connectivity Checks  
| Check | Rule | Threshold |
|---|---|---|
| Snapping tolerance | Line endpoints and devices within this distance are treated as connected | **0.04 ft** |
| Dangle length | Primary line dangles longer than this are flagged | 2.5 ft |
| Disconnected primary | Primary segments not traced back to a source substation | 0 allowed |  
Revision 6 used a snapping tolerance of 0.1 ft. Do not use the Revision 6 value for any conversion that started a

In [15]:
# Chain
chain = rc.rag_prompt | llm | StrOutputParser()
chain.invoke({"context": rc.format_docs(docs_retrieved), "question": question})

'For Prairie Wind Rural Electric, use a **snapping tolerance of 0.25 ft**, which replaces the Revision 7 default of 0.04 ft. This exception was approved by the QA lead and applies to all PWRE feature classes. [08_coop_prairie_wind.md] [04_qa_thresholds.md]'

### RAG chain with sources

Like rag-from-scratch's `rag_chain_with_source`: retrieve once and return the retrieved documents with the answer, so we can check whether the right file was retrieved.

In [16]:
rag_chain = rc.build_rag_chain(retriever, llm)

def ask(q):
    out = rag_chain.invoke(q)
    print(out["answer"])
    print("\nRetrieved:", sorted({d.metadata["source"] for d in out["context"]}))
    return out

_ = ask("For Bluestem Valley, how are service points with source status RET mapped?")

For BVEC, source status `RET` maps to `OPR_STAT_CD` = `OOS`, and the original status is copied into `CNV_NOTE` as `SRC_STAT=RET` [06_coop_bluestem_valley.md]. Service points with status `RET` and no meter number must also be copied into `CNV_HOLD_SVC` for cooperative review before delivery [06_coop_bluestem_valley.md].

Retrieved: ['05_attribute_mapping_general.md', '06_coop_bluestem_valley.md', '11_troubleshooting.md']


In [17]:
# A question the docs don't cover: should abstain
_ = ask("What is the maximum allowed span length between poles for Bluestem Valley?")

not in the docs

Retrieved: ['04_qa_thresholds.md', '06_coop_bluestem_valley.md', '07_coop_ozark_ridge.md']


## Part 5: Baseline (LLM alone)

Same model, no retrieval, two versions:

| Baseline | Prompt |
|---|---|
| `baseline` (guarded) | One-line description of GridLine + the same "say *not in the docs* if you don't know" rule as RAG |
| `baseline_open` | The bare question, nothing else. This is how an analyst would use a general chatbot. |

The guarded baseline tests whether the model admits it doesn't know. The open baseline tests the proposal's claim that a general LLM gives a confident, standard answer that is wrong locally.

In [18]:
baseline_chain = rc.build_baseline_chain(llm)
open_baseline_chain = rc.build_open_baseline_chain(llm)

for q in ["In the GridLine target model, what PHS_CD value is used for a two-phase AC conductor?",
          "For Bluestem Valley, how are service points with source status RET mapped?"]:
    print("Q:", q)
    print("Guarded baseline:", baseline_chain.invoke(q), "\n")
    print("Open baseline:", open_baseline_chain.invoke(q), "\n")

Q: In the GridLine target model, what PHS_CD value is used for a two-phase AC conductor?


Guarded baseline: not in the docs 



Open baseline: I don't have reliable information on the GridLine target model's PHS_CD code list, so I can't give you the exact value for a two-phase AC conductor without guessing.

In most utility phase-coding schemes, a two-phase conductor is recorded as a combination of two phase designations (for example, AB, BC, or CA), but the exact codes and any numeric mappings vary by model. To confirm the value, check the GridLine data dictionary or code-domain table for PHS_CD, or the model's documentation for conductor/phase attributes. If you can share the relevant excerpt, I can help you interpret it. 

Q: For Bluestem Valley, how are service points with source status RET mapped?


Guarded baseline: not in the docs 



Open baseline: I don't have access to a "Bluestem Valley" dataset, so I can't tell you how its RET-status service points are mapped. I don't know whether it's a GIS layer, a utility database, or a data export, or what "RET" stands for in that system.

In water-utility or environmental datasets, "RET" most often means **retired**, referring to a source (such as a well or intake) that is no longer in use. In that case, such points are usually mapped in one of a few ways:

- **Separate symbology**, such as a grey or hollow marker instead of the active-source symbol.
- **Filtered out** of the default view, and shown only when a "show inactive/retired" toggle is on.
- **Linked to the service points** they supplied, so the connection is visible even though the source is inactive.

To give you a precise answer, I'd need one of the following:

- The data dictionary or field definitions for the source status field
- The layer's symbology or legend settings (if it's a GIS project)
- The organiza

## Part 6: Evaluation

10 questions in `eval/questions.json`:

- 4 **local conventions** (Q01, Q02, Q05, Q06)
- 3 **cooperative exceptions** that override a default (Q03, Q04, Q07)
- 2 **not covered** by the docs (Q08, Q09): the correct answer is "not in the docs"
- 1 **public GIS knowledge** control (Q10)

Each covered question has 1–3 key facts. Scoring is automatic, then reviewed by hand:

| Metric | How it's measured |
|---|---|
| Accuracy | Share of key facts found in the answer (covered questions) |
| Hallucination | Flagged if the system answers an uncovered question, or answers a covered one but misses a key fact. Confirmed by hand in `manual_hallucination`. |
| Abstention | Says "not in the docs" on Q08 and Q09; false abstentions on covered questions are counted separately |
| Retrieval hit | The correct source file is among the top-4 chunks (RAG only). Separates retrieval failures from generation failures. |

In [19]:
import json
import pandas as pd
from evaluate import run_eval, summarize

questions = json.loads(rc.EVAL_FILE.read_text(encoding="utf-8"))
pd.DataFrame(questions)[["id", "type", "question", "source"]]

,id,type,question,source
0,Q01,local,"In the GridLine target model, what PHS_CD valu...",02_domain_codes.md
1,Q02,local,What snapping tolerance does the GridLine QA s...,04_qa_thresholds.md
2,Q03,coop_exception,What snapping tolerance should I use for Prair...,08_coop_prairie_wind.md
3,Q04,coop_exception,How should retired poles be handled in the Oza...,07_coop_ozark_ridge.md
4,Q05,local,What value goes in OPR_VOLT_KV when the source...,05_attribute_mapping_general.md
5,Q06,local,"What does QA error GLC-214 mean, and how do I ...",11_troubleshooting.md
6,Q07,coop_exception,"For Bluestem Valley, how are service points wi...",06_coop_bluestem_valley.md
7,Q08,uncovered,What is the maximum allowed span length betwee...,NaN
8,Q09,uncovered,How should water hydrant features be mapped in...,NaN
9,Q10,public_control,What is the EPSG code for WGS 84 geographic co...,03_coordinate_systems.md


In [20]:
results = run_eval(rag_chain, baseline_chain, questions, open_baseline_chain)
results["manual_hallucination"] = ""  # fill in after reading answers.csv

[Q01] In the GridLine target model, what PHS_CD value is used for a two-phase AC conductor?


[Q02] What snapping tolerance does the GridLine QA suite use by default for connectivity checks?


[Q03] What snapping tolerance should I use for Prairie Wind Rural Electric, and why?


[Q04] How should retired poles be handled in the Ozark Ridge conversion?


[Q05] What value goes in OPR_VOLT_KV when the source voltage is blank or can't be parsed?


[Q06] What does QA error GLC-214 mean, and how do I resolve it?


[Q07] For Bluestem Valley, how are service points with source status RET mapped?


[Q08] What is the maximum allowed span length between poles for Bluestem Valley?


[Q09] How should water hydrant features be mapped in a GridLine conversion?


[Q10] What is the EPSG code for WGS 84 geographic coordinates?


### Results by question

In [21]:
pd.set_option("display.max_colwidth", 120)
results.pivot(index="id", columns="system",
              values=["accuracy", "abstained", "hallucination_flag"])

accuracy                    abstained                       \
system baseline baseline_open  rag  baseline baseline_open    rag   
id                                                                  
Q01         0.0           0.0  1.0      True          True  False   
Q02         0.0           0.0  1.0      True          True  False   
Q03         0.0           0.0  1.0      True          True  False   
Q04         0.0           0.0  1.0      True          True  False   
Q05         0.0           0.0  1.0      True          True  False   
Q06         0.0           0.0  1.0      True         False  False   
Q07         0.0           0.0  1.0      True          True  False   
Q08         NaN           NaN  NaN      True          True   True   
Q09         NaN           NaN  NaN      True          True   True   
Q10         1.0           1.0  1.0     False         False  False   

       hallucination_flag                       
system           baseline baseline_open    rag  
id                                              
Q01                 False         False  False  
Q02                 False         False  False  
Q03                 False         False  False  
Q04                 False         False  False  
Q05                 False         False  False  
Q06                 False          True  False  
Q07                 False         False  False  
Q08                 False         False  False  
Q09                 False         False  False  
Q10                 False         False  False

In [22]:
# RAG retrieval: did the right file come back?
results[results.system == "rag"][["id", "type", "retrieval_hit", "retrieved"]]

,id,type,retrieval_hit,retrieved
2,Q01,local,True,"01_target_schema.md, 07_coop_ozark_ridge.md, 04_qa_thresholds.md, 02_domain_codes.md"
5,Q02,local,True,"08_coop_prairie_wind.md, 04_qa_thresholds.md, 09_conversion_tools.md, 11_troubleshooting.md"
8,Q03,coop_exception,True,"08_coop_prairie_wind.md, 04_qa_thresholds.md, 07_coop_ozark_ridge.md"
11,Q04,coop_exception,True,"07_coop_ozark_ridge.md, 06_coop_bluestem_valley.md, 00_conversion_overview.md"
14,Q05,local,True,"05_attribute_mapping_general.md, 08_coop_prairie_wind.md, 07_coop_ozark_ridge.md"
17,Q06,local,True,11_troubleshooting.md
20,Q07,coop_exception,True,"06_coop_bluestem_valley.md, 05_attribute_mapping_general.md, 11_troubleshooting.md"
23,Q08,uncovered,None,"07_coop_ozark_ridge.md, 04_qa_thresholds.md, 06_coop_bluestem_valley.md"
26,Q09,uncovered,None,"01_target_schema.md, 08_coop_prairie_wind.md"
29,Q10,public_control,True,"03_coordinate_systems.md, 07_coop_ozark_ridge.md, 08_coop_prairie_wind.md"


### Summary table: baseline vs. RAG

In [23]:
summary = summarize(results)
summary

,accuracy (key facts),hallucination flags,abstained on uncovered,false abstentions,retrieval hit rate
system,,,,,
baseline,0.125,0,2/2,7,NaN
baseline_open,0.125,1,2/2,6,NaN
rag,1.000,0,2/2,0,1.0


### Read the answers side by side

In [24]:
for qid, g in results.groupby("id"):
    print(f"=== {qid}: {g.question.iloc[0]}")
    for _, r in g.iterrows():
        print(f"[{r.system}] {r.answer}\n")

=== Q01: In the GridLine target model, what PHS_CD value is used for a two-phase AC conductor?
[baseline] not in the docs

[baseline_open] I don't have reliable information about a "GridLine target model" or its PHS_CD (phase code) domain, so I can't give you the exact value without guessing.

In utility data models, phase codes usually list the energized phases as letter combinations, such as A, B, C, AB, BC, AC, or ABC, with N sometimes used for neutral. If GridLine follows that convention, a two-phase AC conductor would be coded with the two phase letters it carries (for example, AB, BC, or AC). Some models instead use numeric or mnemonic codes, though.

To confirm the exact value, check the GridLine data dictionary or code-domain table for PHS_CD, or ask the people who maintain the model. If you can share the documentation excerpt, I can help you interpret it.

[rag] **22** [02_domain_codes.md]

=== Q02: What snapping tolerance does the GridLine QA suite use by default for connecti

In [25]:
rc.RESULTS_DIR.mkdir(exist_ok=True)
results.to_csv(rc.RESULTS_DIR / "answers.csv", index=False)
summary.to_csv(rc.RESULTS_DIR / "summary.csv")
print("Saved to", rc.RESULTS_DIR)

Saved to C:\Users\mills\Geog582_GeoAI\final_project\results


## Part 7 (optional): Multi-Query retrieval

From rag-from-scratch Part 5. The LLM rewrites the question five ways, retrieves for each, and takes the unique union of the results. This can help when an analyst's wording doesn't match the doc's wording (e.g. "tolerance" vs. "snapping"). Compare the retrieval hit rate with Part 6.

In [26]:
from langchain_core.load import dumps, loads

template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines, with no numbering or other text.
Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    | llm
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
)

def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs"""
    flattened = [dumps(doc) for sublist in documents for doc in sublist]
    return [loads(doc) for doc in dict.fromkeys(flattened)]

multi_query_retriever = generate_queries | retriever.map() | get_unique_union

docs_mq = multi_query_retriever.invoke("Which tolerance applies to the paper-digitized co-op?")
sorted({d.metadata["source"] for d in docs_mq})

C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in dict.fromkeys(flattened)]
C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in dict.fromkeys(flattened)]


['02_domain_codes.md',
 '04_qa_thresholds.md',
 '05_attribute_mapping_general.md',
 '06_coop_bluestem_valley.md',
 '08_coop_prairie_wind.md',
 '09_conversion_tools.md']

In [27]:
# Retrieval hit rate: single query vs. multi-query (covered questions only)
rows = []
for item in questions:
    if not item["source"]:
        continue
    single = {d.metadata["source"] for d in retriever.invoke(item["question"])}
    multi = {d.metadata["source"] for d in multi_query_retriever.invoke(item["question"])}
    rows.append({"id": item["id"], "single_hit": item["source"] in single,
                 "multi_hit": item["source"] in multi, "multi_chunks": len(multi)})
mq = pd.DataFrame(rows)
print(mq[["single_hit", "multi_hit"]].mean())
mq

C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in dict.fromkeys(flattened)]


C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in dict.fromkeys(flattened)]


C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in dict.fromkeys(flattened)]


C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in dict.fromkeys(flattened)]


C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in dict.fromkeys(flattened)]


C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in dict.fromkeys(flattened)]


C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in dict.fromkeys(flattened)]


single_hit    1.0
multi_hit     1.0
dtype: float64


C:\Users\mills\AppData\Local\Temp\ipykernel_27316\3583098431.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in dict.fromkeys(flattened)]


,id,single_hit,multi_hit,multi_chunks
0,Q01,True,True,6
1,Q02,True,True,4
2,Q03,True,True,4
3,Q04,True,True,5
4,Q05,True,True,6
5,Q06,True,True,1
6,Q07,True,True,8
7,Q10,True,True,4


## Notes for the report

- **Why RAG here:** the answers are private, change often (QA Revision 6 → 7), and need a traceable source. Fine-tuning would go stale with every doc update.
- **What RAG is not for:** questions about the data itself (feature counts, current values) or spatial reasoning over geometry. Those need database/PostGIS queries.
- **Scale:** the 12-doc corpus would fit in one prompt. Retrieval matters at Confluence scale, where it also keeps cost and latency down.
- **Limits of the test:** 10 questions show behavior; they are not a statistical benchmark. The automatic hallucination flag is a proxy, confirmed by hand.